# 05_integrated_operational_silver

## Purpose

Build an integrated Silver dataset for regional operations.

This notebook joins cleaned Silver grid events with Silver asset reference data and Silver weather context to prepare a reusable base table for Day 4 business logic.

In [0]:
import sys
import importlib

repo_src_path = "/Workspace/Users/dirella@startsteps.org/vattenfall-week9-capstone-anna/src"

if repo_src_path not in sys.path:
    sys.path.append(repo_src_path)

import transforms.integration_transforms as integration_transforms
importlib.reload(integration_transforms)

from validation.silver_checks import print_basic_quality_summary

In [0]:
catalog = "vattenfall_dev"

silver_grid_table = f"{catalog}.refined.silver_grid_events"
silver_asset_table = f"{catalog}.refined.silver_asset_reference"
silver_weather_table = f"{catalog}.refined.silver_weather"

target_table = f"{catalog}.refined.silver_regional_operations_base"

print("Silver grid table:", silver_grid_table)
print("Silver asset reference table:", silver_asset_table)
print("Silver weather table:", silver_weather_table)
print("Target integrated Silver table:", target_table)

In [0]:
grid_df = spark.table(silver_grid_table)
asset_df = spark.table(silver_asset_table)
weather_df = spark.table(silver_weather_table)

print("Grid rows:", grid_df.count())
print("Asset reference rows:", asset_df.count())
print("Weather rows:", weather_df.count())

display(grid_df.limit(10))
display(asset_df.limit(10))
display(weather_df.limit(10))

In [0]:
integrated_df = integration_transforms.build_regional_operations_base(
    grid_df=grid_df,
    asset_df=asset_df,
    weather_df=weather_df
)

print("Integrated rows:", integrated_df.count())

display(integrated_df.limit(20))

In [0]:
(
    integrated_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

print("Written integrated Silver table:", target_table)

In [0]:
silver_df = spark.table(target_table)

print_basic_quality_summary(silver_df, target_table)

display(silver_df.limit(20))

In [0]:
print("Null check for key integrated fields:")

for column_name in ["event_id", "asset_id", "region", "event_day"]:
    null_count = silver_df.filter(f"{column_name} IS NULL").count()
    print(column_name, "nulls:", null_count)

print("Rows with missing asset_name after asset join:")
missing_asset_count = silver_df.filter("asset_name IS NULL").count()
print("missing asset_name:", missing_asset_count)

print("Rows with weather context available:")
weather_context_count = silver_df.filter("temperature_c IS NOT NULL").count()
print("weather context rows:", weather_context_count)